# Tool Persistence

Every tool in `proto_tools` runs inside an isolated virtual environment managed by `ToolInstance`. That isolation is what keeps heavy dependencies like PyTorch and ESM out of your main environment, and it's what lets every tool expose the same clean interface regardless of how different their dependencies are underneath. It has a second consequence, too: by default, every call to a `run_*` function is completely self-contained. A fresh subprocess spins up, the model loads, inference runs, and the subprocess exits. Nothing leaks between calls, and no GPU memory lingers after one finishes.

That safety is usually what you want. For a one-off call in a notebook, paying a single model load is a fine price for getting a clean slate afterwards. But once you're running a batch, whether that's folding dozens of sequences, sweeping hyperparameters, or stepping through an optimizer loop, reloading the model on every call quickly becomes the dominant cost. A forward pass might take a second; loading ESMFold can take forty.

When a workload does call for keeping the model in memory across calls, `ToolInstance` offers a small family of persistence mechanisms. These are peers to the one-shot default, not an upgrade over it; each is the right choice for a different shape of workload. The table below is a quick map of when to reach for each one, and the rest of this guide walks through them in order of increasing control.

| Method | Caches? | Cleanup | Best for |
|---|---|---|---|
| Default (one-shot) | No | Automatic | Single calls, safety-first |
| `ToolInstance.persist()` | Yes, auto | Automatic on exit | Batch jobs, optimization loops |
| `ToolInstance.persist_tool()` | Yes, named tool | Automatic on exit | Multi-GPU, multiple instances |
| `ToolInstance.get()` | Yes, until close | Manual | Long-running sessions |

---

## 1. Default Behavior (One-Shot)

If you don't do anything special, every `run_*` function is one-shot. Each call launches a fresh subprocess, loads the model, runs inference, and tears everything down when it's finished. Isolation is the goal: nothing runs in the background, no GPU memory is held between calls, and one call cannot affect another. For a notebook or a quick script where you only need a single prediction, this is exactly the behavior you want.

The cost becomes visible the moment you issue two calls in a row. Both pay the load in full.

In [ ]:
from proto_tools.tools.structure_prediction.esmfold import (
    run_esmfold, ESMFoldInput, ESMFoldConfig,
)

# Two consecutive calls; each loads the model from scratch
output1 = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda"),
)
# ~38s (model load + inference)

output2 = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda"),
)
# ~38s (model load + inference again)

<img src="assets/tool-persistence/default.svg" alt="Default one-shot: each call reloads the model and releases GPU memory" style={{width: "100%", maxWidth: "560px", display: "block", margin: "1.5rem auto"}} />

That's exactly the behavior you want for one-off calls. For workloads that invoke the same tool repeatedly, reach for one of the persistence modes below instead.

---

## 2. `persist()` Context Manager (Recommended)

The most convenient way to turn on persistence is `ToolInstance.persist()`. It works like `torch.inference_mode()`: wrap the block of code where you want persistence, and any tool called inside is cached on first use and cleaned up automatically when the block exits. You don't have to name the tool upfront, and you don't change the tool-call signature. If a loop calls several tools, each one is cached on first use, so a design loop that runs both ProteinMPNN and ESMFold stays warm on both for the life of the block.

In [ ]:
from proto_tools.utils.tool_instance import ToolInstance

sequences = [
    "MKTLLILAVVAAALA",
    "GAVLTVLLGGLLLA",
    "MGQQPGKVLGDQRR",
    "AAKIKVLGDQRRQA",
]

with ToolInstance.persist():
    for seq in sequences:
        output = run_esmfold(
            ESMFoldInput(complexes=[seq]),
            ESMFoldConfig(device="cuda"),
        )

# Call 1:    ~39s (model load + inference)
# Calls 2-4: ~1s each (inference only)
# Total:     ~42s

<img src="assets/tool-persistence/persist.svg" alt="With persist(): one load, many infers inside the with block" style={{width: "100%", maxWidth: "560px", display: "block", margin: "1.5rem auto"}} />

Inside the `with` block, the first call still pays the full load cost, because the model has to come into memory somehow. Every call after that skips the load entirely and runs inference against the already-warm worker. When the block exits, the worker is shut down, GPU memory is released, and you're back in the default safe state without having to remember to clean anything up.

This is the pattern you want for almost every batch workload: loops over sequences, optimization passes, anything that calls the same tool more than once.

---

## 3. `persist_tool()` for Named Instances

`persist_tool(tool_name)` is a narrower version of `persist()` that scopes persistence to a single named tool. For most batch workloads `persist()` is what you want, since it already caches every tool called inside the block. The reason to reach for `persist_tool()` is when you need more than one live worker for the same tool at the same time.

This comes up most often with multi-GPU setups where you'd like one ESMFold worker pinned to `cuda:0`, another to `cuda:1`, and the ability to route each call to the right one. `persist_tool()` supports this through the `instance_name` argument. Each named instance runs in its own subprocess. At call time, pass the handle you got from the context manager (or just the instance name as a string) as the `instance=` argument on the `run_*` call, and the call is dispatched to that specific worker.

In [ ]:
with ToolInstance.persist_tool("esmfold", instance_name="worker_a") as inst_a:
    with ToolInstance.persist_tool("esmfold", instance_name="worker_b"):

        # Route each call to a specific worker
        out_a = run_esmfold(
            ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
            ESMFoldConfig(device="cuda:0"),
            instance=inst_a,            # pass the instance object
        )

        out_b = run_esmfold(
            ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
            ESMFoldConfig(device="cuda:1"),
            instance="worker_b",        # or pass the instance name
        )

<img src="assets/tool-persistence/named.svg" alt="Named instances: two workers, each pinned to its own GPU and routed by instance name" style={{width: "100%", maxWidth: "560px", display: "block", margin: "1.5rem auto"}} />

Nested `with` blocks keep both workers alive for the duration of the inner block, and the pool is torn down cleanly when the outermost block exits.

---

## 4. Manual Lifecycle with `get()` and `shutdown()`

Sometimes the lifecycle you care about isn't a block at all. In a Jupyter notebook, for instance, you typically want a model to stay warm across many cells, including idle periods, without having to put your entire session inside a single `with` statement. `ToolInstance.get()` gives you direct control: it creates (or fetches) a persistent worker that sticks around until you explicitly shut it down.

In [ ]:
# Create a persistent instance; stays cached until explicitly shut down
tool = ToolInstance.get("esmfold")

for seq in sequences:
    output = run_esmfold(
        ESMFoldInput(complexes=[seq]),
        ESMFoldConfig(device="cuda"),
    )

# Clean up when done; stops worker and evicts from cache
tool.shutdown()

# You can also shut down by name without a reference to the instance
ToolInstance.shutdown_instance("esmfold")

<img src="assets/tool-persistence/get.svg" alt="Manual lifecycle: explicit get() loads the worker; shutdown() releases it on demand" style={{width: "100%", maxWidth: "560px", display: "block", margin: "1.5rem auto"}} />

The contract is simple: after `get()`, the worker is loaded and will serve every subsequent `run_*` call with a matching configuration. Call `tool.shutdown()` when you're done, or `ToolInstance.shutdown_instance("esmfold")` if you don't have the handle nearby. Reach for `get()` when the natural unit of your work is a session rather than a block.

---

## 5. Auto-Restart on Config Changes

Whichever persistence mode you pick, the worker always reflects the configuration it was last loaded with. If you change a load-time parameter, whether that's `device`, `model_checkpoint`, `model_name`, or any other field marked `reload_on_change=True`, the persistence layer detects the mismatch and transparently restarts the worker with the new config before running your call.

This is why `persist()` and `get()` don't take a `device=` argument of their own: they don't need to. The first call inside the block determines the config, and any later change picks itself up without any action from you.

---

## 6. Timeout

Every tool call enforces a timeout, defaulting to 600 seconds (ten minutes). If a single call exceeds the timeout, the subprocess is killed and a `TimeoutError` is raised so the rest of your program can decide what to do.

In [ ]:
# Set a 120-second timeout
output = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda", timeout=120),
)

For persistent workers, the timeout applies per-call. A slow call that trips the timeout does not tear down the worker; subsequent calls continue to run against the already-loaded model as if nothing had happened.

---

## Next Steps

<CardGroup cols={2}>
  <Card title="Device Management" icon="microchip" href="/tools/guides/device-management">
    Control which GPU each tool call runs on.
  </Card>
  <Card title="Parallel Execution" icon="layer-group" href="/tools/guides/parallel-execution">
    Run many tool calls concurrently across workers.
  </Card>
</CardGroup>